# 04 · Consultas analíticas — Athena / Spark SQL
**Tech Challenge Fase 3 · State of Data Brasil (Data Hackers/Bain)**

| Item | Descrição |
|---|---|
| **Objetivo** | Executar as consultas SQL que respondem às perguntas de negócio. Localmente rodam em **Spark SQL**; no AWS, **as mesmas queries** rodam no **Amazon Athena** sobre o Glue Data Catalog. |
| **Origem** | `silver_core`, `silver_serie_longa` (Parquet catalogado) |
| **Requisitos atendidos** | R4 (catalogação), R6 (consultas com Athena/Glue Notebook) |

### DDL de catalogação no AWS (executar uma vez no Athena)
```sql
CREATE DATABASE IF NOT EXISTS stateofdata;

CREATE EXTERNAL TABLE IF NOT EXISTS stateofdata.silver_core (
    id string, idade int, faixa_idade string, genero string, cor_raca string,
    pcd string, uf string, regiao string, nivel_ensino string, area_formacao string,
    situacao_trabalho string, setor string, num_funcionarios string, gestor int,
    cargo_gestor string, cargo string, cargo_grupo string, nivel string,
    faixa_salarial string, salario_pm double, tempo_exp_dados string, satisfeito int,
    entrevistas_6m string, mudar_emprego_6m string, layoff string, layoff_sim int,
    modelo_atual string, modelo_ideal string, modelo_atual_h string, modelo_ideal_h string,
    atitude_presencial string, prioridade_ia string, prioridade_ia_h string, atuacao string
    -- + colunas binárias lang_/cloud_/bi_/genai_/crit_/des_
)
PARTITIONED BY (ano int)
STORED AS PARQUET
LOCATION 's3://<bucket>/datalake/silver/silver_core/';

MSCK REPAIR TABLE stateofdata.silver_core;  -- descobre as partições ano=YYYY
```
> Alternativa recomendada: **Glue Crawler** apontando para `s3://<bucket>/datalake/silver/`
> cataloga as tabelas automaticamente (mesmo efeito, sem DDL manual).

> ⚠️ **Nota de reorganização (padrão Medallion) — leia antes de comparar caminhos.**
>
> O **código** das células abaixo já aponta para a estrutura atual do repositório:
> `../../datalake/{bronze,silver,gold}`, `../../consumption/charts` e as tabelas Gold
> renomeadas em inglês (`gold_roles`, `gold_salary_by_seniority`, …).
>
> As **saídas gravadas** neste arquivo, porém, são as da execução original, anterior à
> reorganização: elas ainda exibem os caminhos e rótulos antigos
> (`/home/claude/tech_challenge/datalake`, `gold_cargos`, `G02_cargos_2025`). Foram mantidas
> intactas de propósito — são evidência do que efetivamente rodou, e reescrevê-las à mão
> falsearia o registro de execução. **Os números não mudam**; apenas os rótulos de caminho.
>
> Para regravar as saídas com a nomenclatura nova, reexecute os notebooks na ordem 01 → 05
> a partir de `src/notebooks/` (requer PySpark; o notebook 05 requer apenas pandas/matplotlib).


In [1]:
# ============================================================
# PORTABILIDADE AWS GLUE (descomente APENAS no Glue Notebook)
# ============================================================
# %glue_version 4.0
# %worker_type G.1X
# %number_of_workers 2
# %idle_timeout 30
# No AWS, ajuste BASE para: s3://<seu-bucket>/datalake
# ============================================================

In [2]:
# -*- coding: utf-8 -*-
"""Consultas analíticas SQL sobre Silver/Gold.
Localmente executam via Spark SQL; no AWS Academy Lab, as mesmas queries
rodam no Amazon Athena sobre as tabelas do Glue Data Catalog."""
from pyspark.sql import SparkSession

BASE = "../../datalake"
spark = (SparkSession.builder.master("local[2]").appName("consultas_analiticas")
         .config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Views equivalentes às tabelas do Glue Data Catalog
spark.read.parquet(f"{BASE}/silver/silver_core").createOrReplaceTempView("silver_core")
spark.read.parquet(f"{BASE}/silver/silver_serie_longa").createOrReplaceTempView("silver_serie_longa")

In [3]:
# Q1 — Estrutura do mercado: top cargos por edição (Pergunta 1)
# Nota de portabilidade: ROW_NUMBER em subconsulta (em vez de QUALIFY) para
# compatibilidade total com Glue 4.0/5.0 (Spark 3.x) e Amazon Athena.
print("Q1 — Top 5 grupos de cargo por edição")
spark.sql("""
    SELECT ano, cargo_grupo, n, pct
    FROM (
        SELECT ano, cargo_grupo, COUNT(*) AS n,
               ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY ano), 1) AS pct,
               ROW_NUMBER() OVER (PARTITION BY ano ORDER BY COUNT(*) DESC) AS rk
        FROM silver_core
        WHERE cargo_grupo IS NOT NULL
        GROUP BY ano, cargo_grupo
    )
    WHERE rk <= 5
    ORDER BY ano, n DESC
""").show(15, False)


Q1 — Top 5 grupos de cargo por edição


+----+-------------------------------+---+----+
|ano |cargo_grupo                    |n  |pct |
+----+-------------------------------+---+----+
|2023|Análise de Dados               |907|23.5|
|2023|Ciência de Dados               |687|17.8|
|2023|Engenharia/Arquitetura de Dados|684|17.7|
|2023|Business Intelligence          |506|13.1|
|2023|Outros                         |404|10.5|
|2024|Análise de Dados               |957|25.1|
|2024|Ciência de Dados               |687|18.0|
|2024|Engenharia/Arquitetura de Dados|661|17.3|
|2024|Business Intelligence          |396|10.4|
|2024|Outros                         |362|9.5 |
|2025|Análise de Dados               |599|24.0|
|2025|Engenharia/Arquitetura de Dados|430|17.2|
|2025|Ciência de Dados               |424|17.0|
|2025|Outros                         |287|11.5|
|2025|Business Intelligence          |215|8.6 |
+----+-------------------------------+---+----+



In [4]:
# Q2 — Escada salarial: mediana do ponto médio por senioridade e ano (Pergunta 2)
print("Q2 — Salário mediano (PM) por senioridade e ano")
spark.sql("""
    SELECT ano, nivel, COUNT(*) AS n,
           ROUND(PERCENTILE_APPROX(salario_pm, 0.5), 0) AS salario_mediano_pm
    FROM silver_core
    WHERE salario_pm IS NOT NULL AND nivel IS NOT NULL
    GROUP BY ano, nivel
    ORDER BY ano, salario_mediano_pm
""").show(12, False)

Q2 — Salário mediano (PM) por senioridade e ano


+----+-------------------+----+------------------+
|ano |nivel              |n   |salario_mediano_pm|
+----+-------------------+----+------------------+
|2023|Júnior             |1046|3501.0            |
|2023|Pleno              |1392|7001.0            |
|2023|Sênior             |1419|10001.0           |
|2024|Júnior             |868 |3501.0            |
|2024|Pleno              |1377|7001.0            |
|2024|Sênior             |1573|14001.0           |
|2025|Júnior             |518 |3501.0            |
|2025|Pleno              |776 |7001.0            |
|2025|Sênior             |858 |14001.0           |
|2025|Especialista/Staff+|349 |18001.0           |
+----+-------------------+----+------------------+



In [5]:
# Q3 — Gap salarial por gênero controlando senioridade (Pergunta 3)
print("Q3 — Gap salarial F/M por senioridade — 2025/26")
spark.sql("""
    WITH sal AS (
        SELECT nivel, genero,
               PERCENTILE_APPROX(salario_pm, 0.5) AS mediana, COUNT(*) AS n
        FROM silver_core
        WHERE ano = 2025 AND salario_pm IS NOT NULL
          AND nivel IS NOT NULL AND genero IN ('Masculino','Feminino')
        GROUP BY nivel, genero)
    SELECT f.nivel,
           ROUND(f.mediana, 0) AS mediana_feminino,
           ROUND(m.mediana, 0) AS mediana_masculino,
           ROUND(100 * (f.mediana / m.mediana - 1), 1) AS gap_pct,
           f.n AS n_feminino, m.n AS n_masculino
    FROM sal f JOIN sal m ON f.nivel = m.nivel
    WHERE f.genero = 'Feminino' AND m.genero = 'Masculino'
    ORDER BY mediana_masculino
""").show(10, False)

Q3 — Gap salarial F/M por senioridade — 2025/26


+-------------------+----------------+-----------------+-------+----------+-----------+
|nivel              |mediana_feminino|mediana_masculino|gap_pct|n_feminino|n_masculino|
+-------------------+----------------+-----------------+-------+----------+-----------+
|Júnior             |3501.0          |3501.0           |0.0    |146       |367        |
|Pleno              |7001.0          |7001.0           |0.0    |174       |599        |
|Sênior             |10001.0         |14001.0          |-28.6  |178       |675        |
|Especialista/Staff+|14001.0         |18001.0          |-22.2  |70        |276        |
+-------------------+----------------+-----------------+-------+----------+-----------+



In [6]:
# Q4 — Tecnologias: evolução de Python, SQL e AWS (Pergunta 4)
print("Q4 — Evolução de Python, SQL e AWS (série longa, 6 edições)")
spark.sql("""
    SELECT ano,
           ROUND(100 * AVG(lang_python), 1) AS pct_python,
           ROUND(100 * AVG(lang_sql), 1)    AS pct_sql,
           ROUND(100 * AVG(cloud_aws), 1)   AS pct_aws,
           COUNT(lang_python) AS base_valida
    FROM silver_serie_longa
    GROUP BY ano ORDER BY ano
""").show(10, False)

Q4 — Evolução de Python, SQL e AWS (série longa, 6 edições)


+----+----------+-------+-------+-----------+
|ano |pct_python|pct_sql|pct_aws|base_valida|
+----+----------+-------+-------+-----------+
|2019|44.4      |40.5   |23.3   |1765       |
|2021|72.5      |80.1   |42.4   |1857       |
|2022|70.1      |79.4   |38.7   |2981       |
|2023|74.9      |83.7   |30.6   |3772       |
|2024|81.8      |87.7   |44.1   |3589       |
|2025|92.0      |84.2   |48.3   |2096       |
+----+----------+-------+-------+-----------+



In [7]:
# Q5 — IA generativa como prioridade (Pergunta 5 — tese central)
print("Q5 — IA generativa como prioridade da empresa (visão dos gestores)")
spark.sql("""
    SELECT ano, prioridade_ia_h, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY ano), 1) AS pct
    FROM silver_core
    WHERE prioridade_ia_h IS NOT NULL
    GROUP BY ano, prioridade_ia_h
    ORDER BY ano, prioridade_ia_h
""").show(20, False)

Q5 — IA generativa como prioridade da empresa (visão dos gestores)


+----+----------------------------------+---+----+
|ano |prioridade_ia_h                   |n  |pct |
+----+----------------------------------+---+----+
|2023|1. Principal prioridade da empresa|110|12.3|
|2023|2. Entre as principais (2-4 anos) |214|23.9|
|2023|3. Iniciativas isoladas, sem foco |275|30.7|
|2023|4. Não é prioridade               |258|28.8|
|2023|5. Não sabe opinar                |39 |4.4 |
|2024|1. Principal prioridade da empresa|235|22.5|
|2024|2. Entre as principais (2-4 anos) |325|31.1|
|2024|3. Iniciativas isoladas, sem foco |301|28.8|
|2024|4. Não é prioridade               |155|14.8|
|2024|5. Não sabe opinar                |29 |2.8 |
|2025|1. Principal prioridade da empresa|155|23.8|
|2025|2. Entre as principais (2-4 anos) |240|36.8|
|2025|3. Iniciativas isoladas, sem foco |169|25.9|
|2025|4. Não é prioridade               |74 |11.3|
|2025|5. Não sabe opinar                |14 |2.1 |
+----+----------------------------------+---+----+



In [8]:
# Q6 — Modelo de trabalho: gap entre praticado e desejado (Pergunta 6)
print("Q6 — Modelo de trabalho atual vs ideal — 2025/26")
spark.sql("""
    SELECT COALESCE(a.modelo, i.modelo) AS modelo, a.pct_atual, i.pct_ideal,
           ROUND(i.pct_ideal - a.pct_atual, 1) AS gap_pp
    FROM (SELECT modelo_atual_h AS modelo,
                 ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_atual
          FROM silver_core WHERE ano = 2025 AND modelo_atual_h IS NOT NULL
          GROUP BY modelo_atual_h) a
    FULL JOIN (SELECT modelo_ideal_h AS modelo,
                 ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_ideal
          FROM silver_core WHERE ano = 2025 AND modelo_ideal_h IS NOT NULL
          GROUP BY modelo_ideal_h) i
    ON a.modelo = i.modelo
    ORDER BY gap_pp DESC
""").show(10, False)

Q6 — Modelo de trabalho atual vs ideal — 2025/26


+--------------------+---------+---------+------+
|modelo              |pct_atual|pct_ideal|gap_pp|
+--------------------+---------+---------+------+
|Híbrido flexível    |19.5     |43.0     |23.5  |
|100% remoto         |39.7     |42.1     |2.4   |
|Híbrido (dias fixos)|20.0     |13.0     |-7.0  |
|100% presencial     |20.8     |1.9      |-18.9 |
+--------------------+---------+---------+------+



In [9]:
# Q7 — Termômetro do mercado: layoff, satisfação e intenção de troca (Pergunta 7)
print("Q7 — Termômetro do mercado por edição")
spark.sql("""
    SELECT ano,
           ROUND(100 * AVG(satisfeito), 1)  AS pct_satisfeitos,
           ROUND(100 * AVG(layoff_sim), 1)  AS pct_layoff_sim,
           COUNT(satisfeito) AS n_validos
    FROM silver_core
    GROUP BY ano ORDER BY ano
""").show(10, False)

print("OK — consultas executadas.")
spark.stop()

Q7 — Termômetro do mercado por edição


+----+---------------+--------------+---------+
|ano |pct_satisfeitos|pct_layoff_sim|n_validos|
+----+---------------+--------------+---------+
|2023|72.0           |32.2          |4753     |
|2024|68.6           |28.6          |4863     |
|2025|69.0           |24.9          |3228     |
+----+---------------+--------------+---------+

OK — consultas executadas.
